# S08 — Solutions: Testing, Type Hints, Exceptions & Defensive Programming

**Python for Applied Physics** | Master in Applied Physics  
⚠️ *Instructor reference — do not distribute before the exercise session.*

In [ ]:
from __future__ import annotations
import numpy as np
import pytest
import warnings
import logging
import h5py, tempfile, os
from typing import Optional

# Shared exception hierarchy
class PhysicsError(Exception): pass
class PulseError(PhysicsError, ValueError):
    def __init__(self, parameter, value, reason):
        self.parameter = parameter; self.value = value
        super().__init__(f"{parameter}={value}: {reason}")

# Shared GaussianPulse
class GaussianPulse:
    c = 3e8
    def __init__(self, tau: float, wavelength: float, energy: float) -> None:
        if tau <= 0:        raise PulseError('tau', tau, 'must be positive')
        if wavelength <= 0: raise PulseError('wavelength', wavelength, 'must be positive')
        if energy < 0:      raise PulseError('energy', energy, 'must be non-negative')
        self._tau = tau; self._wavelength = wavelength; self._energy = energy
    @property
    def tau(self) -> float: return self._tau
    @property
    def wavelength(self) -> float: return self._wavelength
    @property
    def energy(self) -> float: return self._energy
    @property
    def fwhm(self) -> float: return 2*np.sqrt(np.log(2))*self._tau
    @property
    def bandwidth(self) -> float: return np.sqrt(np.log(2))/(np.pi*self._tau)
    def intensity(self, t: np.ndarray) -> np.ndarray: return np.exp(-t**2/self._tau**2)
    def spectrum(self, t: np.ndarray):
        N=len(t); dt=t[1]-t[0]
        E_f=np.fft.fftshift(np.fft.fft(np.sqrt(self.intensity(t))))
        return np.fft.fftshift(np.fft.fftfreq(N,d=dt)), np.abs(E_f)**2
    def __repr__(self) -> str:
        return f"GaussianPulse(tau={self._tau*1e15:.1f} fs, wavelength={self._wavelength*1e9:.0f} nm, energy={self._energy*1e6:.1f} µJ)"

# Shared OpticalElement classes
class OpticalElement:
    def matrix(self) -> np.ndarray: raise NotImplementedError
    def propagate(self, ray: np.ndarray) -> np.ndarray: return self.matrix() @ ray
    def __matmul__(self, other):
        if isinstance(other, OpticalElement): return _Composed(self.matrix() @ other.matrix())
        return NotImplemented
class _Composed(OpticalElement):
    def __init__(self, M): self._M = M
    def matrix(self): return self._M
class FreeSpace(OpticalElement):
    def __init__(self, d: float) -> None:
        if d < 0: raise ValueError(f"d must be non-negative, got {d}")
        self.d = d
    def matrix(self) -> np.ndarray: return np.array([[1.0, self.d],[0.0,1.0]])
class ThinLens(OpticalElement):
    def __init__(self, f: float) -> None:
        if f == 0: raise ValueError("f cannot be zero")
        self.f = f
    def matrix(self) -> np.ndarray: return np.array([[1.0,0.0],[-1/self.f,1.0]])
class Mirror(OpticalElement):
    def __init__(self, R: float) -> None: self.R = R
    def matrix(self) -> np.ndarray: return np.array([[1.0,0.0],[-2/self.R,1.0]])

print("Setup complete")

---
## Exercise 1 — Type-annotated `shared/optics.py`

In [ ]:
def gaussian_intensity(r: np.ndarray, w: float, P: float) -> np.ndarray:
    """Radial intensity profile of a Gaussian beam (W/m²)."""
    I0 = 2 * P / (np.pi * w**2)
    return I0 * np.exp(-2 * r**2 / w**2)

def gaussian_field_2d(
    x: np.ndarray, y: np.ndarray, w: float, E0: float = 1.0
) -> np.ndarray:
    """2D Gaussian field via broadcasting."""
    X = x[np.newaxis, :]; Y = y[:, np.newaxis]
    return E0 * np.exp(-(X**2 + Y**2) / w**2)

def free_space(d: float) -> np.ndarray:
    """ABCD matrix for free-space propagation."""
    return np.array([[1.0, d],[0.0, 1.0]])

def thin_lens(f: float) -> np.ndarray:
    """ABCD matrix for a thin lens."""
    return np.array([[1.0, 0.0],[-1.0/f, 1.0]])

def fresnel_rs(
    n1: float, n2: float, theta_i: float | np.ndarray
) -> float | np.ndarray:
    """s-polarisation Fresnel reflection coefficient."""
    sin_t = n1 / n2 * np.sin(theta_i)
    cos_t = np.sqrt(np.clip(1 - sin_t**2, 0, None))
    cos_i = np.cos(theta_i)
    return (n1*cos_i - n2*cos_t) / (n1*cos_i + n2*cos_t)

r = np.linspace(-1e-3, 1e-3, 64)
I = gaussian_intensity(r, 300e-6, 1.0)
assert I.shape == r.shape
print("Type-annotated functions verified ✓")
print("Note: wrong type does not raise at runtime — hints are for tooling only.")

---
## Exercise 2 — Unit tests for `gaussian_intensity`

In [ ]:
def test_gaussian_intensity_peak():
    w, P = 500e-6, 1.0
    I0   = gaussian_intensity(np.array([0.0]), w=w, P=P)[0]
    assert I0 == pytest.approx(2*P/(np.pi*w**2), rel=1e-6)

def test_gaussian_intensity_at_w():
    w, P = 500e-6, 1.0
    I0   = 2*P/(np.pi*w**2)
    I_w  = gaussian_intensity(np.array([w]), w=w, P=P)[0]
    assert I_w == pytest.approx(I0*np.exp(-2), rel=1e-6)

def test_gaussian_intensity_symmetry():
    w, P = 500e-6, 1.0
    r    = np.linspace(-3*w, 3*w, 256)
    I    = gaussian_intensity(r, w=w, P=P)
    assert np.allclose(I, I[::-1])

def test_gaussian_intensity_integrated_power():
    w, P  = 500e-6, 1.0
    r     = np.linspace(0, 5*w, 10000)
    I     = gaussian_intensity(r, w=w, P=P)
    P_int = 2*np.pi*np.trapz(I*r, r)
    assert P_int == pytest.approx(P, rel=1e-3)

def test_gaussian_intensity_linear_in_power():
    w = 500e-6
    r = np.array([0.0])
    I1 = gaussian_intensity(r, w, P=1.0)[0]
    I2 = gaussian_intensity(r, w, P=2.0)[0]
    assert I2 == pytest.approx(2*I1, rel=1e-6)

def test_gaussian_intensity_shape():
    r = np.linspace(-1e-3, 1e-3, 128)
    I = gaussian_intensity(r, w=300e-6, P=1.0)
    assert I.shape == r.shape

for fn in [test_gaussian_intensity_peak, test_gaussian_intensity_at_w,
           test_gaussian_intensity_symmetry, test_gaussian_intensity_integrated_power,
           test_gaussian_intensity_linear_in_power, test_gaussian_intensity_shape]:
    fn(); print(f"  {fn.__name__} ✓")
print("All passed ✓")

---
## Exercise 3 — Parametrized TBP test

In [ ]:
TBP_LIMIT = 2*np.log(2)/np.pi

def compute_tbp(tau_fs: float, wavelength_nm: float = 800.0) -> float:
    tau = tau_fs * 1e-15
    N, dt = 4096, 5e-15
    t    = (np.arange(N) - N//2) * dt
    p    = GaussianPulse(tau=tau, wavelength=wavelength_nm*1e-9, energy=50e-6)
    I_t  = p.intensity(t)
    freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
    S    = np.abs(np.fft.fftshift(np.fft.fft(np.sqrt(I_t))))**2
    def fwhm(x, y):
        yn=y/y.max(); above=yn>=0.5
        return x[above].max()-x[above].min()
    return fwhm(t, I_t) * fwhm(freq, S)

@pytest.mark.parametrize('tau_fs', [30,50,80,100,150,200,500])
def test_tbp_vs_duration(tau_fs: float):
    assert compute_tbp(float(tau_fs)) == pytest.approx(TBP_LIMIT, rel=0.01)

@pytest.mark.parametrize('wavelength_nm', [800.0, 1030.0])
@pytest.mark.parametrize('tau_fs', [100, 200])
def test_tbp_vs_wavelength(tau_fs, wavelength_nm):
    assert compute_tbp(float(tau_fs), wavelength_nm) == pytest.approx(TBP_LIMIT, rel=0.01)

for tau_fs in [30,50,80,100,150,200,500]:
    tbp = compute_tbp(float(tau_fs))
    print(f"  tau={tau_fs:4d} fs: TBP={tbp:.4f} ✓")
print("All passed ✓")

---
## Exercise 4 — Test suite for `GaussianPulse`

In [ ]:
# conftest.py
@pytest.fixture
def standard_pulse() -> GaussianPulse:
    return GaussianPulse(tau=100e-15, wavelength=800e-9, energy=50e-6)

@pytest.fixture
def time_axis() -> np.ndarray:
    N, dt = 4096, 5e-15
    return (np.arange(N) - N//2) * dt


# test_pulse_classes.py
def test_repr_contains_tau(p): assert '100' in repr(p)
def test_repr_contains_wavelength(p): assert '800' in repr(p)
def test_intensity_peak(p): assert p.intensity(np.array([0.0]))[0] == pytest.approx(1.0)
def test_intensity_at_tau(p):
    assert p.intensity(np.array([p.tau]))[0] == pytest.approx(np.exp(-1), rel=1e-6)
def test_raises_negative_tau():
    with pytest.raises(PulseError): GaussianPulse(-1e-15, 800e-9, 0.0)
def test_raises_zero_wavelength():
    with pytest.raises(PulseError): GaussianPulse(100e-15, 0.0, 0.0)
def test_raises_negative_energy():
    with pytest.raises(PulseError): GaussianPulse(100e-15, 800e-9, -1e-6)
def test_zero_energy_valid():
    p = GaussianPulse(100e-15, 800e-9, 0.0); assert p.energy == 0.0
def test_tbp(p, t):
    assert p.fwhm * p.bandwidth == pytest.approx(2*np.log(2)/np.pi, rel=0.01)
def test_spectrum_shape(p, t):
    freq, S = p.spectrum(t); assert S.shape == t.shape
def test_spectrum_non_negative(p, t):
    _, S = p.spectrum(t); assert np.all(S >= 0)

sp = GaussianPulse(100e-15, 800e-9, 50e-6)
ta = (np.arange(4096)-2048)*5e-15

for fn, args in [
    (test_repr_contains_tau, (sp,)), (test_repr_contains_wavelength, (sp,)),
    (test_intensity_peak, (sp,)), (test_intensity_at_tau, (sp,)),
    (test_raises_negative_tau, ()), (test_raises_zero_wavelength, ()),
    (test_raises_negative_energy, ()), (test_zero_energy_valid, ()),
    (test_tbp, (sp, ta)), (test_spectrum_shape, (sp, ta)),
    (test_spectrum_non_negative, (sp, ta)),
]:
    fn(*args); print(f"  {fn.__name__} ✓")
print("All passed ✓")

---
## Exercise 5 — Test the `OpticalElement` hierarchy

In [ ]:
def test_free_space_identity():
    assert np.allclose(FreeSpace(0.0).matrix(), np.eye(2))

def test_collimation():
    f = 0.1
    sys = FreeSpace(f) @ ThinLens(f) @ FreeSpace(f)
    ray_out = sys.propagate(np.array([0.0, np.radians(5)]))
    assert ray_out[1] == pytest.approx(0.0, abs=1e-10)

@pytest.mark.parametrize('element', [FreeSpace(0.1), ThinLens(0.2), Mirror(0.5)])
def test_det_equals_one(element):
    assert np.linalg.det(element.matrix()) == pytest.approx(1.0, rel=1e-10)

def test_composition_associativity():
    A, B, C = FreeSpace(0.1), ThinLens(0.2), FreeSpace(0.3)
    M1 = ((A @ B) @ C).matrix()
    M2 = (A @ (B @ C)).matrix()
    assert np.allclose(M1, M2)

def test_free_space_negative_raises():
    with pytest.raises(ValueError): FreeSpace(-0.1)

def test_thin_lens_zero_raises():
    with pytest.raises(ValueError): ThinLens(0.0)

test_free_space_identity(); print("  test_free_space_identity ✓")
test_collimation();         print("  test_collimation ✓")
test_composition_associativity(); print("  test_composition_associativity ✓")
test_free_space_negative_raises(); print("  test_free_space_negative_raises ✓")
test_thin_lens_zero_raises(); print("  test_thin_lens_zero_raises ✓")
for el in [FreeSpace(0.1), ThinLens(0.2), Mirror(0.5)]:
    test_det_equals_one(el); print(f"  test_det_equals_one({el.__class__.__name__}) ✓")
print("All passed ✓")

---
## Exercise 6 — Custom exceptions + defensive guards

In [ ]:
class PulseIOError(PhysicsError, OSError): pass

logger_io = logging.getLogger('shared.io')

def save_pulse_hdf5(
    path: str,
    t: np.ndarray,
    E_t: np.ndarray,
    compression: str = 'gzip',
    compress: Optional[bool] = None,
    **metadata,
) -> None:
    # Guard: file extension
    if not path.endswith('.h5'):
        raise ValueError(f"path must end in .h5, got {path!r}")
    # Guard: matching lengths
    if len(t) != len(E_t):
        raise ValueError(f"t and E_t must have the same length: {len(t)} vs {len(E_t)}")
    # Deprecation
    if compress is not None:
        warnings.warn(
            "The 'compress' parameter is deprecated; use 'compression' instead.",
            DeprecationWarning, stacklevel=2,
        )
    # Existing file warning
    if os.path.exists(path):
        logger_io.warning("File already exists and will be overwritten: %s", path)

    logger_io.debug("Writing pulse to %s (N=%d)", path, len(t))

    with h5py.File(path, 'w') as f:
        grp = f.create_group('pulse')
        grp.create_dataset('t',   data=t,   compression=compression)
        grp.create_dataset('E_t', data=E_t, compression=compression)
        grp.create_dataset('I_t', data=np.abs(E_t)**2)
        for key, value in metadata.items():
            grp.attrs[key] = value

    logger_io.info("Saved pulse: %s", path)


N = 512; dt = 5e-15
t_test = (np.arange(N)-N//2)*dt
E_test = np.exp(-t_test**2/(2*(100e-15)**2))

def test_save_wrong_extension():
    with pytest.raises(ValueError, match=r'\.h5'):
        save_pulse_hdf5('output.txt', t_test, E_test)

def test_save_mismatched_lengths():
    with pytest.raises(ValueError, match='length'):
        with tempfile.TemporaryDirectory() as tmp:
            save_pulse_hdf5(os.path.join(tmp,'p.h5'), t_test, E_test[:10])

def test_save_deprecation_warning():
    with tempfile.TemporaryDirectory() as tmp:
        with pytest.warns(DeprecationWarning):
            save_pulse_hdf5(os.path.join(tmp,'p.h5'), t_test, E_test, compress=True)

def test_save_roundtrip():
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp,'pulse.h5')
        save_pulse_hdf5(path, t_test, E_test, tau_fs=100.0)
        with h5py.File(path,'r') as f:
            assert np.allclose(f['pulse']['t'][:], t_test)
            assert np.allclose(f['pulse']['E_t'][:], E_test)
            assert f['pulse'].attrs['tau_fs'] == pytest.approx(100.0)

for fn in [test_save_wrong_extension, test_save_mismatched_lengths,
           test_save_deprecation_warning, test_save_roundtrip]:
    fn(); print(f"  {fn.__name__} ✓")
print("All passed ✓")

---
## Exercise 7 — Coverage gap-filling tests

In [ ]:
def test_gaussian_intensity_single_point():
    I = gaussian_intensity(np.array([0.0]), w=300e-6, P=1.0)
    assert I.shape == (1,)
    assert I[0] > 0

def test_gaussian_intensity_zero_power():
    r = np.linspace(-1e-3, 1e-3, 64)
    I = gaussian_intensity(r, w=300e-6, P=0.0)
    assert np.all(I == 0.0)

def test_free_space_zero_distance():
    assert np.allclose(FreeSpace(0.0).matrix(), np.eye(2))

def test_pulse_very_short():
    p = GaussianPulse(1e-16, 800e-9, 0.0)   # 100 as
    assert p.tau == pytest.approx(1e-16)

def test_thin_lens_negative_focal_length():
    """Diverging lens (f < 0) is valid."""
    tl = ThinLens(-0.1)
    M  = tl.matrix()
    assert M[1, 0] == pytest.approx(10.0)   # -1/f = -1/-0.1 = 10

def test_pulse_spectrum_dc():
    """Spectrum of real Gaussian is symmetric — peak at DC."""
    p    = GaussianPulse(100e-15, 800e-9, 50e-6)
    t    = (np.arange(4096)-2048)*5e-15
    freq, S = p.spectrum(t)
    idx_dc  = np.argmin(np.abs(freq))
    assert S[idx_dc] == S.max()

for fn in [test_gaussian_intensity_single_point, test_gaussian_intensity_zero_power,
           test_free_space_zero_distance, test_pulse_very_short,
           test_thin_lens_negative_focal_length, test_pulse_spectrum_dc]:
    fn(); print(f"  {fn.__name__} ✓")

print("\nAll gap-filling tests passed ✓")
print("Run from terminal: pytest tests/ --cov=shared --cov-fail-under=80")